In [33]:
import pandas as pd
import numpy as np
import psycopg2
import sqlalchemy as db
from sqlalchemy import create_engine
import yaml

In [34]:
with open('C:\\Users\\dylan\\OneDrive\\Documentos\\mensajeria\\config.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_co = config['mensajeria']
    config_etl = config['ETL_PRO']

# Construct the database URL
url_co = (f"{config_co['drivername']}://{config_co['user']}:{config_co['password']}@{config_co['host']}:"
          f"{config_co['port']}/{config_co['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")
# Create the SQLAlchemy Engine
mensajeria = create_engine(url_co)
etl_conn = create_engine(url_etl)

In [35]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)
tabla_servicios = pd.read_sql_table('mensajeria_servicio', mensajeria)
tabla_estados=pd.read_sql_table('mensajeria_estadosservicio', mensajeria)
tabla_sede = pd.read_sql_table('clientes_usuarioaquitoy', mensajeria)

tabla_servicios = tabla_servicios[tabla_servicios['descripcion_cancelado'].isnull()]
tabla_servicios.rename(columns={'id':'servicio_id'}, inplace=True)

tabla_servicios["key_trans_servicio"] = range(1,len(tabla_servicios)+1 )


tabla_servicios.drop(columns=['novedades', 'fecha_deseada', 'hora_deseada','nombre_recibe','telefono_recibe','hora_visto_por_mensajero',
                              'descripcion_pago','ida_y_regreso','tipo_pago_id','prioridad','visto_por_mensajero','multiples_origenes',
                              'descripcion_cancelado','descripcion_multiples_origenes','activo','asignar_mensajero','es_prueba',
                              'nombre_solicitante','descripcion','origen_id','destino_id','tipo_vehiculo_id','tipo_servicio_id'], inplace=True)



tabla_servicios = tabla_servicios.merge(tabla_sede[['cliente_id','user_id','sede_id']],
                                        left_on=['cliente_id','usuario_id'], right_on=['cliente_id','user_id'], how='left')

tabla_servicios['mensajero_id']=tabla_servicios['mensajero_id'].fillna(0).astype(int)
tabla_servicios['mensajero_id']=tabla_servicios['mensajero2_id'].fillna(0).astype(int)
tabla_servicios['mensajero_id']=tabla_servicios['mensajero3_id'].fillna(0).astype(int)
tabla_servicios.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 346 entries, 0 to 345
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   servicio_id         346 non-null    int64         
 1   fecha_solicitud     346 non-null    datetime64[ns]
 2   hora_solicitud      346 non-null    object        
 3   cliente_id          346 non-null    int64         
 4   mensajero_id        346 non-null    int64         
 5   usuario_id          346 non-null    int64         
 6   ciudad_destino_id   346 non-null    int64         
 7   ciudad_origen_id    346 non-null    int64         
 8   mensajero2_id       72 non-null     float64       
 9   mensajero3_id       10 non-null     float64       
 10  key_trans_servicio  346 non-null    int64         
 11  user_id             76 non-null     float64       
 12  sede_id             76 non-null     float64       
dtypes: datetime64[ns](1), float64(4), int64(7), object

In [36]:
tabla_estados_pivot_fecha = tabla_estados.pivot_table(
    index='servicio_id',
    columns='estado_id',
    values='fecha',
    aggfunc='first'      
).reset_index()
tabla_estados_pivot_fecha.head(9)

tabla_estados_pivot_hora = tabla_estados.pivot_table(
    index='servicio_id',
    columns='estado_id',
    values='hora',
    aggfunc='first'      
).reset_index()
trans_servicios = tabla_servicios.merge(tabla_estados_pivot_fecha, on='servicio_id', how='left')
trans_servicios.drop(columns=[3], inplace=True)
trans_servicios.rename(columns={1:'fecha_iniciado', 2:'fecha_mensajero_asignado', 4:'fecha_recogido_mensajero',
                                 5:'fecha_entregado',6:'fecha_finalizado_completo'}, inplace=True)
trans_servicios = trans_servicios.merge(tabla_estados_pivot_hora, on='servicio_id', how='left')
trans_servicios.drop(columns=[3], inplace=True)
trans_servicios.rename(columns={1:'hora_iniciado', 2:'hora_mensajero_asignado', 4:'hora_recogido_mensajero',
                                 5:'hora_entregado',6:'hora_finalizado_completo'}, inplace=True)
trans_servicios.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 346 entries, 0 to 345
Data columns (total 23 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   servicio_id                346 non-null    int64         
 1   fecha_solicitud            346 non-null    datetime64[ns]
 2   hora_solicitud             346 non-null    object        
 3   cliente_id                 346 non-null    int64         
 4   mensajero_id               346 non-null    int64         
 5   usuario_id                 346 non-null    int64         
 6   ciudad_destino_id          346 non-null    int64         
 7   ciudad_origen_id           346 non-null    int64         
 8   mensajero2_id              72 non-null     float64       
 9   mensajero3_id              10 non-null     float64       
 10  key_trans_servicio         346 non-null    int64         
 11  user_id                    76 non-null     float64       
 12  sede_id 

In [37]:
trans_servicios.to_sql('trans_servicio', etl_conn, if_exists='replace', index=False)



346